# Notebook 7: Gradient Descent

**Difficulty:** ⭐⭐⭐⭐ | **Estimated Time:** 3 hours

**Theme:** Predicting house sale prices

---

## Why Does This Matter?

Ordinary Least Squares (OLS) gives us an exact solution: **θ = (XᵀX)⁻¹Xᵀy**. So why bother with gradient descent?

The answer is **scale**. Inverting XᵀX costs **O(p³)** — cubic in the number of features. That is manageable when p = 10 or even p = 1,000. But:
- An NLP model with a vocabulary of 100,000 words has p ≈ 100,000 features
- A ResNet image model has p ≈ 25,000,000 parameters
- A large language model (GPT-4 scale) has p ≈ 1,000,000,000,000 parameters

Computing (XᵀX)⁻¹ for even the NLP case would require inverting a 100,000 × 100,000 matrix — **that is 10¹⁵ floating-point operations**. It simply does not fit in memory, let alone finish in a reasonable time.

**Gradient descent** sidesteps the matrix inversion entirely. It finds the minimum iteratively — taking small steps downhill on the loss surface — and scales to billions of parameters.

---

## The Real-World Analogy: Hiking Down a Mountain Blindfolded

Imagine you are dropped on a mountainside in thick fog. Your goal is to reach the lowest valley. You cannot see the full landscape — you can only feel the slope under your feet right now.

Your strategy:
1. Feel which direction is steepest downhill at your current position
2. Take a step in that direction
3. Repeat until the ground feels flat (you have converged)

In this analogy:
- **Your position** = the model weights (θ)
- **Your altitude** = the loss (how wrong the model currently is)
- **The slope under your feet** = the gradient ∇L(θ)
- **Step size** = the learning rate α
- **The valley** = the minimum loss (optimal weights)

The key insight: **you never need to see the whole mountain**. You only need the local slope at each step. That is why gradient descent scales where OLS does not.

## Section 1: The Gradient — Plain English and Formula

### What Is a Gradient?

The **gradient** ∇L(θ) is a vector that points in the direction of **steepest increase** in the loss function. Each component tells you: *"If I increase this weight slightly, does the loss go up or down, and by how much?"*

Plain English: "The gradient tells us how each weight is contributing to the error right now."

Since we want to **decrease** loss, we move in the **opposite** direction:

$$\theta \leftarrow \theta - \alpha \nabla L(\theta)$$

### Deriving the Gradient for Linear Regression

Our loss function is Mean Squared Error (MSE):

$$L(\theta) = \frac{1}{n} \| X\theta - y \|^2 = \frac{1}{n} (X\theta - y)^\top (X\theta - y)$$

Expanding and taking the derivative with respect to θ:

$$\frac{\partial L}{\partial \theta} = \frac{2}{n} X^\top (X\theta - y)$$

**Step-by-step derivation:**
1. Let **r = Xθ - y** (the residual vector, shape n×1)
2. L = (1/n) rᵀr = (1/n) Σᵢ rᵢ²
3. ∂L/∂θⱼ = (2/n) Σᵢ rᵢ · ∂rᵢ/∂θⱼ = (2/n) Σᵢ rᵢ · Xᵢⱼ
4. In matrix form: ∂L/∂θ = (2/n) Xᵀr = (2/n) Xᵀ(Xθ - y)

**Intuition:** Each weight θⱼ is updated proportional to how correlated the residuals are with feature j. If feature j consistently predicts in the wrong direction, its weight gets pulled back.

In [ ]:
# ── Imports and global style settings ──────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from matplotlib import cm
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

# Consistent visual theme across all plots
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'font.size': 10
})

# Fix random seed for reproducibility
np.random.seed(42)

print("Libraries loaded successfully.")

## Section 2: Generate Synthetic House Price Data

We will use a simple linear relationship: **price ≈ 150 × size + noise**

- Feature x: house size in hundreds of square feet (e.g., 10 = 1,000 sq ft)
- Target y: sale price in thousands of dollars
- True relationship: price = 50 + 150 × size (with Gaussian noise)

This synthetic data lets us know the *true* optimum so we can verify gradient descent finds it.

In [ ]:
# ── Generate synthetic house price data ────────────────────────────────────
n_samples = 200

# House size in hundreds of sq ft, uniformly between 5 and 30 (500–3,000 sq ft)
X_raw = np.random.uniform(5, 30, n_samples)

# True parameters: intercept=50k, slope=150k per 100 sq ft
true_intercept = 50.0
true_slope = 150.0

# Add Gaussian noise (realistic market noise)
noise = np.random.normal(0, 40, n_samples)
y = true_intercept + true_slope * X_raw + noise

# Add bias column (column of 1s) for the intercept term
# X shape: (n_samples, 2) — column 0 = ones, column 1 = size
X = np.column_stack([np.ones(n_samples), X_raw])

# Also keep standardized version for gradient descent (important for convergence)
scaler = StandardScaler()
X_scaled_inner = scaler.fit_transform(X_raw.reshape(-1, 1))
X_scaled = np.column_stack([np.ones(n_samples), X_scaled_inner.ravel()])

print(f"Dataset: {n_samples} houses")
print(f"House size range: {X_raw.min():.1f} to {X_raw.max():.1f} (hundreds sq ft)")
print(f"Price range: ${y.min():.0f}k to ${y.max():.0f}k")
print(f"True theta: intercept={true_intercept}, slope={true_slope}")

# Quick plot to visualize the data
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(X_raw, y, alpha=0.4, edgecolors='steelblue', facecolors='lightblue',
           linewidth=0.5, label='Observed sales')
ax.set_xlabel('House Size (hundreds sq ft)')
ax.set_ylabel('Sale Price ($1,000s)')
ax.set_title('Synthetic House Price Dataset')
ax.legend()
plt.tight_layout()
plt.show()

## Section 3: Batch Gradient Descent — From Scratch

**Algorithm:**
```
Initialize θ randomly (or zeros)
For each iteration t = 1, 2, ..., T:
    predictions = X @ θ          # shape (n,)
    residuals   = predictions - y # shape (n,)
    gradient    = (2/n) * Xᵀ @ residuals  # shape (p,)
    θ           = θ - α * gradient
    record loss = mean(residuals²)
Stop when |loss_t - loss_{t-1}| < tolerance
```

"Batch" means we use **all n training examples** to compute each gradient update.

In [ ]:
# ── Batch Gradient Descent implementation ──────────────────────────────────
def batch_gradient_descent(X, y, learning_rate=0.01, n_iterations=1000, tol=1e-6):
    """
    Batch GD for linear regression.
    Returns: theta (final weights), loss_history (list of MSE per iteration)
    """
    n, p = X.shape
    theta = np.zeros(p)          # start at zero weights
    loss_history = []

    for i in range(n_iterations):
        residuals = X @ theta - y           # predictions minus ground truth
        loss = np.mean(residuals ** 2)      # MSE over all n samples
        loss_history.append(loss)

        # Gradient: (2/n) * Xᵀ(Xθ - y)
        gradient = (2 / n) * X.T @ residuals
        theta = theta - learning_rate * gradient   # parameter update

        # Early stopping: convergence check
        if i > 0 and abs(loss_history[-2] - loss_history[-1]) < tol:
            print(f"Converged at iteration {i+1} (loss change < {tol})")
            break

    return theta, loss_history

# Run on standardized features (critical: GD converges much faster on scaled data)
theta_gd, loss_history = batch_gradient_descent(
    X_scaled, y, learning_rate=0.01, n_iterations=2000
)

# Compare to sklearn's OLS solution for validation
reg_ols = LinearRegression().fit(X_scaled_inner, y)
theta_ols = np.array([reg_ols.intercept_, reg_ols.coef_[0]])

print(f"\nGradient Descent theta: intercept={theta_gd[0]:.2f}, slope={theta_gd[1]:.2f}")
print(f"OLS (sklearn) theta:    intercept={theta_ols[0]:.2f}, slope={theta_ols[1]:.2f}")
print(f"Final GD MSE: {loss_history[-1]:.2f}")

## Section 4: Learning Curve and Line-Fitting Snapshots

In [ ]:
# ── Plot 1: Loss vs Iteration (Learning Curve) ─────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: full learning curve
axes[0].plot(loss_history, color='steelblue', linewidth=1.5)
axes[0].set_xlabel('Iteration')
axes[0].set_ylabel('MSE Loss')
axes[0].set_title('Learning Curve: Loss vs Iteration')
axes[0].annotate(f'Final loss: {loss_history[-1]:.1f}',
                 xy=(len(loss_history)-1, loss_history[-1]),
                 xytext=(len(loss_history)*0.6, loss_history[0]*0.5),
                 arrowprops=dict(arrowstyle='->', color='red'),
                 color='red', fontsize=9)

# Right: log scale to see the tail behavior
axes[1].plot(loss_history, color='darkorange', linewidth=1.5)
axes[1].set_yscale('log')
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('MSE Loss (log scale)')
axes[1].set_title('Learning Curve (Log Scale)')

plt.suptitle('Batch Gradient Descent — Convergence', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# ── Plot 2: Line fitting snapshots at different iterations ─────────────────
# Re-run GD but capture theta at specific iteration snapshots
snapshot_iters = [1, 5, 20, 50, 100, 200, 500, 1000]
snapshots = {}  # iteration -> theta

n, p = X_scaled.shape
theta_snap = np.zeros(p)
lr_snap = 0.01

for i in range(1001):
    residuals = X_scaled @ theta_snap - y
    gradient = (2 / n) * X_scaled.T @ residuals
    theta_snap = theta_snap - lr_snap * gradient
    if i in snapshot_iters:
        snapshots[i] = theta_snap.copy()

# Plot house size vs price with fitted lines at each snapshot
fig, ax = plt.subplots(figsize=(10, 6))

# Scatter plot of data
ax.scatter(X_raw, y, alpha=0.3, color='lightgray', edgecolors='gray',
           linewidth=0.5, zorder=1, label='Data')

# X_raw sorted for smooth line drawing
x_line = np.linspace(X_raw.min(), X_raw.max(), 200)
x_line_scaled = (x_line - scaler.mean_[0]) / scaler.scale_[0]
x_line_design = np.column_stack([np.ones(200), x_line_scaled])

# Color gradient from red (early) to blue (later iterations)
colors = plt.cm.RdYlBu(np.linspace(0.1, 0.9, len(snapshot_iters)))

for (it, th), col in zip(sorted(snapshots.items()), colors):
    y_pred_line = x_line_design @ th
    ax.plot(x_line, y_pred_line, color=col, linewidth=1.8,
            label=f'Iter {it}', alpha=0.85, zorder=2)

ax.set_xlabel('House Size (hundreds sq ft)')
ax.set_ylabel('Sale Price ($1,000s)')
ax.set_title('Line Fitting Progress: Gradient Descent Snapshots')
ax.legend(loc='upper left', fontsize=8, ncol=2)
plt.tight_layout()
plt.show()
print("Early iterations (red): poor fit. Later iterations (blue): converged fit.")

## Section 5: Learning Rate — Too Small, Just Right, Too Large

The learning rate α is the single most important hyperparameter in gradient descent.

| Learning Rate | Effect | Symptom |
|---|---|---|
| α too small (0.0001) | Crawls toward minimum | Loss decreases but takes 10,000+ steps |
| α just right (0.01) | Smooth, efficient convergence | Loss decreases steadily and levels off |
| α too large (1.0) | Overshoots minimum | Loss oscillates or explodes to infinity |

Geometrically, α determines **how far you step** along the gradient direction. A large α means jumping across the valley rather than descending into it.

In [ ]:
# ── Learning Rate Comparison ───────────────────────────────────────────────
learning_rates = {
    'Too slow (α=0.0001)': 0.0001,
    'Just right (α=0.01)': 0.01,
    'Too fast (α=0.8)': 0.8
}

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
line_styles = [('steelblue', '-'), ('green', '-'), ('crimson', '-')]

for ax, (label, lr), (color, ls) in zip(axes, learning_rates.items(), line_styles):
    n_iters = 1000
    n_loc, p_loc = X_scaled.shape
    theta_loc = np.zeros(p_loc)
    losses_loc = []

    for _ in range(n_iters):
        res = X_scaled @ theta_loc - y
        mse = np.mean(res ** 2)
        # Cap loss at 1e8 to prevent inf/nan from breaking the plot
        losses_loc.append(min(mse, 1e8))
        grad = (2 / n_loc) * X_scaled.T @ res
        theta_loc = theta_loc - lr * grad
        if np.any(np.isnan(theta_loc)):   # diverged: stop early
            losses_loc += [1e8] * (n_iters - len(losses_loc))
            break

    ax.plot(losses_loc, color=color, linewidth=1.5)
    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('MSE Loss')
    # Use log scale only when loss is positive and finite
    if min(losses_loc) > 0 and lr < 0.5:
        ax.set_yscale('log')
    ax.set_ylim(bottom=0, top=min(max(losses_loc)*1.1, 1e8))

plt.suptitle('Effect of Learning Rate on Convergence', fontsize=14)
plt.tight_layout()
plt.show()

print("Key takeaway: learning rate must be chosen carefully.")
print("Common strategy: start with α=0.01 and adjust based on the learning curve.")

## Section 6: Visualizing the Loss Surface

For a 2-parameter model (intercept θ₀ and slope θ₁), we can plot the MSE loss as a 2D surface or contour map. This reveals **why** gradient descent works: for linear regression, the loss surface is a **convex paraboloid** — a bowl shape with exactly one global minimum and no local minima.

This is guaranteed by the math: MSE = (1/n)||Xθ - y||² is a quadratic function of θ, and all quadratics in multiple variables that have a minimum are convex. On a convex surface, any downhill path reaches the global minimum.

In [ ]:
# ── Loss Surface: 2D Contour + Gradient Descent Path ──────────────────────
# Compute loss over a grid of (theta_0, theta_1) values
# Using scaled features so the loss surface is well-conditioned

# Grid resolution
n_grid = 80
theta0_range = np.linspace(theta_gd[0] - 200, theta_gd[0] + 200, n_grid)
theta1_range = np.linspace(theta_gd[1] - 200, theta_gd[1] + 200, n_grid)
T0, T1 = np.meshgrid(theta0_range, theta1_range)

# Compute MSE for every (theta0, theta1) pair in the grid
Z = np.zeros_like(T0)
for i in range(n_grid):
    for j in range(n_grid):
        th = np.array([T0[i, j], T1[i, j]])
        res = X_scaled @ th - y
        Z[i, j] = np.mean(res ** 2)

# Record the GD path
path_theta = [np.zeros(2)]   # start at origin
th_path = np.zeros(2)
lr_path = 0.01
for _ in range(300):
    res = X_scaled @ th_path - y
    grad = (2 / n) * X_scaled.T @ res
    th_path = th_path - lr_path * grad
    path_theta.append(th_path.copy())

path_arr = np.array(path_theta)

# Plot the contour map and the GD path
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left: filled contour map
contour = axes[0].contourf(T0, T1, Z, levels=40, cmap='viridis')
fig.colorbar(contour, ax=axes[0], label='MSE Loss')
# Overlay gradient descent path
axes[0].plot(path_arr[:, 0], path_arr[:, 1], 'w-o', markersize=2,
             linewidth=1.2, alpha=0.8, label='GD path')
axes[0].plot(path_arr[0, 0], path_arr[0, 1], 'rs', markersize=8, label='Start')
axes[0].plot(theta_gd[0], theta_gd[1], 'r*', markersize=12, label='Minimum')
axes[0].set_xlabel('θ₀ (intercept)')
axes[0].set_ylabel('θ₁ (slope)')
axes[0].set_title('MSE Loss Contours with GD Path')
axes[0].legend(fontsize=8)

# Right: 3D surface for intuition
ax3d = fig.add_subplot(122, projection='3d')
ax3d.remove()  # replace the flat axes with a 3D one
ax3d = fig.add_axes([0.55, 0.05, 0.42, 0.88], projection='3d')
ax3d.plot_surface(T0, T1, Z, cmap='viridis', alpha=0.7, linewidth=0)
ax3d.plot(path_arr[:, 0], path_arr[:, 1],
          [np.mean((X_scaled @ th - y)**2) for th in path_arr],
          'r-', linewidth=2, label='GD path')
ax3d.set_xlabel('θ₀')
ax3d.set_ylabel('θ₁')
ax3d.set_zlabel('MSE')
ax3d.set_title('MSE Loss Surface (Bowl Shape)')
ax3d.view_init(elev=30, azim=225)

plt.suptitle('The MSE Loss Surface is Convex — One Global Minimum', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()
print("The convex bowl guarantees gradient descent finds the GLOBAL minimum.")

## Section 7: Stochastic Gradient Descent (SGD)

**Batch GD:** computes gradient over ALL n examples → one update per pass.

**Stochastic GD:** computes gradient over **ONE randomly chosen** example → n updates per pass.

| Property | Batch GD | SGD |
|---|---|---|
| Gradient quality | Exact (low variance) | Noisy (high variance) |
| Updates per epoch | 1 | n |
| Speed per epoch | Slow (uses all data) | Fast (uses 1 sample) |
| Memory | High (needs full X) | Low (needs 1 row) |
| Convergence path | Smooth | Noisy / zigzag |
| Escapes local optima | Poor | Better (noise helps) |

The noise in SGD is actually useful in deep learning — it acts as implicit regularization and helps escape shallow local minima.

In [ ]:
# ── Stochastic Gradient Descent (SGD) ─────────────────────────────────────
def sgd(X, y, learning_rate=0.001, n_epochs=30):
    """
    SGD: one gradient step per random sample.
    Returns theta and per-step loss (very noisy).
    """
    n, p = X.shape
    theta = np.zeros(p)
    loss_history = []
    indices = np.arange(n)

    for epoch in range(n_epochs):
        np.random.shuffle(indices)       # shuffle data each epoch
        for i in indices:
            xi = X[i:i+1]                # single sample (shape 1×p)
            yi = y[i:i+1]
            res = xi @ theta - yi
            # Gradient from single sample: 2 * xᵢᵀ * residual
            grad = 2 * xi.T @ res
            theta = theta - learning_rate * grad.ravel()
            # Record full-dataset MSE for fair comparison
            full_res = X @ theta - y
            loss_history.append(np.mean(full_res ** 2))

    return theta, loss_history

theta_sgd, loss_sgd = sgd(X_scaled, y, learning_rate=0.01, n_epochs=20)

print(f"SGD final theta: intercept={theta_sgd[0]:.2f}, slope={theta_sgd[1]:.2f}")
print(f"GD  final theta: intercept={theta_gd[0]:.2f},  slope={theta_gd[1]:.2f}")

## Section 8: Mini-Batch Gradient Descent

Mini-batch GD is the **standard** approach in practice. It splits the data into small batches (typically 32, 64, or 128 samples) and computes the gradient over each batch.

- **Better gradient estimates** than SGD (less variance)
- **Faster than Batch GD** (more updates per epoch)
- **GPU-efficient**: batches of 32–256 saturate GPU memory bandwidth optimally
- **One epoch** = one complete pass through all batches

In [ ]:
# ── Mini-Batch Gradient Descent ────────────────────────────────────────────
def mini_batch_gd(X, y, learning_rate=0.01, n_epochs=30, batch_size=32):
    """
    Mini-batch GD: compute gradient on batches of `batch_size` samples.
    Returns theta and per-epoch loss.
    """
    n, p = X.shape
    theta = np.zeros(p)
    epoch_losses = []
    indices = np.arange(n)

    for epoch in range(n_epochs):
        np.random.shuffle(indices)
        # Split into batches of size `batch_size`
        for start in range(0, n, batch_size):
            batch_idx = indices[start:start + batch_size]
            X_batch = X[batch_idx]
            y_batch = y[batch_idx]
            b = len(batch_idx)             # actual batch size (last batch may be smaller)
            res = X_batch @ theta - y_batch
            grad = (2 / b) * X_batch.T @ res
            theta = theta - learning_rate * grad

        # Compute full MSE at end of each epoch for fair comparison
        full_res = X @ theta - y
        epoch_losses.append(np.mean(full_res ** 2))

    return theta, epoch_losses

# Run all three variants with epoch-level comparison
n_epochs = 50

# Batch GD: one update per epoch → loss per epoch
theta_bgd, _ = batch_gradient_descent(X_scaled, y, 0.01, n_epochs, tol=0)
# Re-run capturing per-epoch loss properly
n_loc, p_loc = X_scaled.shape
th_b = np.zeros(p_loc)
batch_epoch_losses = []
for _ in range(n_epochs):
    res = X_scaled @ th_b - y
    batch_epoch_losses.append(np.mean(res**2))
    grad = (2 / n_loc) * X_scaled.T @ res
    th_b = th_b - 0.01 * grad

theta_mb, mb_epoch_losses = mini_batch_gd(X_scaled, y, 0.01, n_epochs, batch_size=32)

# SGD epoch losses (average over n per-step losses within each epoch)
n_loc2, p_loc2 = X_scaled.shape
th_s = np.zeros(p_loc2)
sgd_epoch_losses = []
for epoch in range(n_epochs):
    idx = np.random.permutation(n_loc2)
    for i in idx:
        xi = X_scaled[i:i+1]; yi = y[i:i+1]
        res = xi @ th_s - yi
        grad = 2 * xi.T @ res
        th_s = th_s - 0.005 * grad.ravel()
    full_res = X_scaled @ th_s - y
    sgd_epoch_losses.append(np.mean(full_res**2))

# ── Convergence Comparison Plot ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
epochs = np.arange(1, n_epochs + 1)
ax.plot(epochs, batch_epoch_losses, 'b-', linewidth=2, label='Batch GD (smooth)')
ax.plot(epochs, mb_epoch_losses, 'g--', linewidth=2, label='Mini-Batch GD (b=32)')
ax.plot(epochs, sgd_epoch_losses, 'r:', linewidth=1.5, alpha=0.8, label='SGD (noisy)')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('Convergence Comparison: Batch vs Mini-Batch vs SGD')
ax.legend()
ax.set_yscale('log')
plt.tight_layout()
plt.show()

print("Batch GD: smoothest convergence. SGD: noisiest. Mini-batch: best trade-off.")
print(f"Final MSE — Batch: {batch_epoch_losses[-1]:.1f}, "
      f"Mini-Batch: {mb_epoch_losses[-1]:.1f}, SGD: {sgd_epoch_losses[-1]:.1f}")

## Section 9: Convergence Criteria and Epochs

**How do you know when to stop?**

Common stopping conditions:
1. **Max iterations:** stop after T iterations (safe but may stop too early or too late)
2. **Loss tolerance:** stop when |L_t - L_{t-1}| < ε (e.g., ε = 1e-6)
3. **Gradient norm:** stop when ||∇L|| < ε (gradient is nearly zero at the minimum)
4. **Validation loss:** stop when validation loss starts increasing (early stopping — prevents overfitting)

**Epoch:** one full pass through the entire training dataset.
- Batch GD: 1 gradient update per epoch
- Mini-batch (b=32, n=200): 200/32 ≈ 7 gradient updates per epoch
- SGD: n=200 gradient updates per epoch

In [ ]:
# ── Gradient norm convergence diagnostic ──────────────────────────────────
n_loc, p_loc = X_scaled.shape
theta_diag = np.zeros(p_loc)
lr_diag = 0.01

loss_diag, grad_norm_diag = [], []

for i in range(500):
    res = X_scaled @ theta_diag - y
    loss_diag.append(np.mean(res**2))
    grad = (2 / n_loc) * X_scaled.T @ res
    grad_norm_diag.append(np.linalg.norm(grad))  # Euclidean norm of gradient
    theta_diag = theta_diag - lr_diag * grad

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(loss_diag, color='steelblue')
axes[0].axhline(y=min(loss_diag), linestyle='--', color='red', alpha=0.6,
                label=f'Minimum MSE ≈ {min(loss_diag):.1f}')
axes[0].set_xlabel('Iteration'); axes[0].set_ylabel('MSE')
axes[0].set_title('Loss Convergence'); axes[0].legend()

axes[1].plot(grad_norm_diag, color='darkorange')
axes[1].axhline(y=1e-3, linestyle='--', color='red', alpha=0.6,
                label='Gradient norm < 0.001 → converged')
axes[1].set_xlabel('Iteration'); axes[1].set_ylabel('||∇L||')
axes[1].set_title('Gradient Norm (Convergence Diagnostic)')
axes[1].set_yscale('log'); axes[1].legend()

plt.suptitle('Diagnosing Convergence', fontsize=13)
plt.tight_layout()
plt.show()

print(f"Gradient norm at start: {grad_norm_diag[0]:.2f}")
print(f"Gradient norm at end:   {grad_norm_diag[-1]:.6f}")
print("Near-zero gradient norm confirms we've reached (near) the minimum.")

## Section 10: Momentum and Adam — Brief Overview

### Momentum

Plain GD can be slow to traverse long, flat valleys in the loss surface (common in deep networks). **Momentum** accumulates a velocity vector in directions of persistent gradient, like a ball rolling downhill that gains speed:

$$v \leftarrow \beta v + (1 - \beta) \nabla L(\theta)$$
$$\theta \leftarrow \theta - \alpha v$$

Typical β = 0.9. This dampens oscillation and accelerates convergence along shallow dimensions.

### Adam (Adaptive Moment Estimation)

**Adam** (Kingma & Ba, 2015) is the default optimizer for neural networks. It maintains:
- **First moment** (mean of gradients) — like momentum
- **Second moment** (uncentered variance of gradients) — adapts learning rate per parameter

Parameters that receive large gradients get a smaller effective learning rate. Parameters with small gradients get a larger effective learning rate. This handles sparse features and varied gradient magnitudes automatically.

**For linear regression:** plain batch GD works perfectly well. Adam and momentum shine in deep networks with millions of parameters, non-convex loss surfaces, and sparse gradients.

In [ ]:
# ── Momentum GD (illustrative implementation) ──────────────────────────────
def momentum_gd(X, y, learning_rate=0.01, beta=0.9, n_iterations=300):
    """
    Gradient descent with momentum.
    beta: momentum coefficient (typical: 0.9)
    """
    n, p = X.shape
    theta = np.zeros(p)
    velocity = np.zeros(p)   # accumulated velocity (starts at 0)
    loss_history = []

    for _ in range(n_iterations):
        res = X @ theta - y
        loss_history.append(np.mean(res**2))
        grad = (2 / n) * X.T @ res
        # Update velocity: blend old velocity with new gradient
        velocity = beta * velocity + (1 - beta) * grad
        theta = theta - learning_rate * velocity

    return theta, loss_history

# Compare plain GD vs Momentum on the same data
_, loss_plain = batch_gradient_descent(X_scaled, y, 0.01, 300, tol=0)
_, loss_momentum = momentum_gd(X_scaled, y, 0.01, 0.9, 300)

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(loss_plain, label='Plain GD', color='steelblue', linewidth=2)
ax.plot(loss_momentum, label='Momentum GD (β=0.9)', color='darkorange',
        linewidth=2, linestyle='--')
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE Loss (log scale)')
ax.set_yscale('log')
ax.set_title('Plain GD vs Momentum GD — Convergence Speed')
ax.legend()
plt.tight_layout()
plt.show()

print("Momentum often converges faster, especially in high-dimensional settings.")
print("For linear regression, the difference is minor — both converge to the same minimum.")

## Section 11: Putting It Together — Gradient Descent vs OLS Summary

In [ ]:
# ── Summary Comparison Table ───────────────────────────────────────────────
summary = pd.DataFrame({
    'Method': ['OLS (Normal Equation)', 'Batch GD', 'Mini-Batch GD', 'SGD'],
    'Complexity per Step': ['O(np²) + O(p³)', 'O(np)', 'O(bp)', 'O(p)'],
    'Requires Learning Rate': ['No', 'Yes', 'Yes', 'Yes'],
    'Scales to Large p': ['No (p > 10k problematic)', 'Yes', 'Yes', 'Yes'],
    'Convergence': ['Exact (1 step)', 'Smooth', 'Slightly noisy', 'Very noisy'],
    'Best For': ['Small datasets, p < 10k', 'Small to medium', 'Deep learning standard', 'Online/streaming data']
})

print(summary.to_string(index=False))

print("\n" + "="*65)
print("GRADIENT DESCENT KEY EQUATIONS SUMMARY")
print("="*65)
print("Update rule:   θ ← θ - α · ∇L(θ)")
print("MSE gradient:  ∇L(θ) = (2/n) · Xᵀ(Xθ - y)")
print("Convergence:   stop when ||∇L|| < ε or |L_t - L_{t-1}| < ε")
print("\nFinal Results:")
print(f"  GD theta:  intercept={theta_gd[0]:.3f}, slope={theta_gd[1]:.3f}")
print(f"  OLS theta: intercept={theta_ols[0]:.3f}, slope={theta_ols[1]:.3f}")
print(f"  Difference: {np.abs(theta_gd - theta_ols).max():.4f} (nearly identical)")

## Self-Check Questions

Test your understanding. Try to answer each question before revealing the solution.

---

**Question 1:** You are running gradient descent and the loss suddenly spikes from 0.3 to 10,000 after 50 iterations. What happened?

<details>
<summary>Reveal Answer</summary>

The learning rate α is too large. When α is too large, the gradient update overshoots the minimum: the model jumps past the bottom of the loss bowl and lands on the other, higher side. From there it jumps again — even further. This runaway process causes the loss to grow exponentially ("divergence"). 

**Fix:** Reduce α by a factor of 10 (e.g., from 0.1 → 0.01) and restart. A well-chosen learning rate produces a smoothly decreasing loss curve.

**Diagnostic check:** If the loss at any step is larger than the initial loss, your learning rate is almost certainly too large.

</details>

---

**Question 2:** Batch GD uses all n examples per step; SGD uses 1. Which is faster per epoch? Which has lower variance in gradient estimates?

<details>
<summary>Reveal Answer</summary>

- **Faster per epoch:** SGD is faster per epoch because it performs n parameter updates using n cheap single-sample gradients, whereas Batch GD performs 1 update using 1 expensive full-dataset gradient. Each SGD update costs O(p); each Batch GD update costs O(np). Total work per epoch is similar, but SGD makes n times more progress in weight space.

- **Lower gradient variance:** Batch GD has lower variance. The full-dataset gradient is the *exact* gradient of the loss. The single-sample SGD gradient is an unbiased but very noisy estimate — it can point in wildly different directions from step to step. Mini-batch GD (b=32–128) offers a practical middle ground.

</details>

---

**Question 3:** For linear regression, gradient descent always finds the global minimum. Why is this true? (Hint: what is the shape of the MSE loss surface?)

<details>
<summary>Reveal Answer</summary>

The MSE loss for linear regression is a **convex quadratic function** of θ:

$$L(\theta) = \frac{1}{n} \|X\theta - y\|^2$$

This is a quadratic form in θ, which defines a **paraboloid** in parameter space — a smooth bowl shape with exactly one flat bottom. A convex function has no local minima other than the global minimum. Therefore:
- Any downhill direction leads toward the global minimum
- Gradient descent cannot get stuck in a local minimum
- There are no saddle points that could trap the algorithm

This is unique to linear regression. For neural networks, the loss surface is non-convex with many local minima and saddle points — convergence guarantees are much weaker.

</details>

---

**Question 4:** What does the learning rate α represent geometrically on the loss surface?

<details>
<summary>Reveal Answer</summary>

Geometrically, α determines **how far you step** along the gradient direction (the direction of steepest descent) on the loss surface.

- **α small:** you take tiny steps — you will eventually reach the minimum but may need thousands of iterations
- **α large:** you take big steps — you might leap over the minimum entirely and land on the opposite slope (higher loss)
- **α optimal:** your steps are sized to efficiently descend without overshooting

More precisely, α scales the gradient vector: the update is Δθ = -α · ∇L(θ). The gradient gives the *direction* of steepest descent; α gives the *magnitude* of the step in that direction. If the loss surface is steep (large gradient), even a small α produces a noticeable step. If the surface is nearly flat (small gradient), a larger α is needed to make progress.

</details>